In [0]:
#Complex SQL 2 | find new and repeat customers | SQL Interview Questions

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DateType
from datetime import date

# Define schema
schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_date", DateType(), True),
    StructField("order_amount", IntegerType(), True)
])

# Data
data = [
    (1, 100, date(2022, 1, 1), 2000),
    (2, 200, date(2022, 1, 1), 2500),
    (3, 300, date(2022, 1, 1), 2100),
    (4, 100, date(2022, 1, 2), 2000),
    (5, 400, date(2022, 1, 2), 2200),
    (6, 500, date(2022, 1, 2), 2700),
    (7, 100, date(2022, 1, 3), 3000),
    (8, 400, date(2022, 1, 3), 1000),
    (9, 600, date(2022, 1, 3), 3000)
]

# Create DataFrame
customer_orders_df = spark.createDataFrame(data, schema)

# Save as table
customer_orders_df.write.mode("overwrite").saveAsTable("customer_orders")

# Display table
display(spark.table("customer_orders"))

In [0]:
spark.sql("""
    with cte as (
        select *, 
            MIN(order_date) OVER(partition by customer_id) as first_visit_date
        from customer_orders 
    )
    select order_date, sum(New) as New, sum(Repeat) as Repeat from (
    select *,
        CASE WHEN order_date = first_visit_date THEN 1 ELSE 0 END as New,
        CASE WHEN order_date != first_visit_date THEN 1 ELSE 0 END as Repeat
    from  cte)

    group by order_date
    
""").display()

In [0]:
from pyspark.sql import functions as F

df = spark.table("customer_orders")
from pyspark.sql.window import Window
w = Window.partitionBy("customer_id")
df_cte = df.withColumn("first_visit_date", F.min("order_date").over(w))

df_flagged = df_cte.withColumn(
    "New", F.when(F.col("order_date") == F.col("first_visit_date"), 1).otherwise(0)
).withColumn(
    "Repeat", F.when(F.col("order_date") != F.col("first_visit_date"), 1).otherwise(0)
)

# Group by order_date and aggregate
result = df_flagged.groupBy("order_date").agg(
    F.sum("New").alias("New"),
    F.sum("Repeat").alias("Repeat")
)

display(result)